# SLT Notebook 5 — Effective theory

**Singular Learning Theory, Iliad Intensive 2026**

Worked solutions, and the output of every code cell, are on the reading page [`slt5_effective_theory.html`](slt5_effective_theory.html) (open it in a web browser).

*Question:* at finite $n$, how singular does a model look, and how much of function space does the free energy actually see?

As in the lecture, $n$ is a parameter scaling the population loss, not a sample count. It is related to generalization from $n$ samples through Watanabe's formula, and we ignore sample noise throughout.

**What to do first.** Two exercises carry the lecture: **5.2a** (a cosine integral, and the truncation theorem it gives) and **5.3a** (Newton polygons in one parameter). Do those, then pick from the rest.

- [5.1](#slt5-s1): 5.1a is optional, for anyone who hasn't seen why generic functions are non-singular. 5.1b is the two-plateau picture from the lecture (lower priority).
- [5.2](#slt5-s2): 5.2a is the core. 5.2b (optional) treats other activations through their nearest complex pole. 5.2c–d are low priority: do §5.3 first.
- [5.3](#slt5-s3): 5.3a is the core. 5.3b adds a second parameter and shows where counting by sizes fails; 5.3c is Furman's cubic model at finite $n$.
- [5.4](#slt5-s4): an optional sheet on tropical arithmetic, the algebra behind Newton polygons.

The learning-zero challenge that goes with the second lecture is in Notebook 6.

**Notation.** Inputs $x\sim\gamma=\mathcal N(0,1)$; function space $\mathcal F=L^2(\gamma)$ with $\langle f,g\rangle=\mathbb E_{x\sim\gamma}[f(x)g(x)]$. A model is a map $\theta\mapsto f_\theta\in\mathcal F$, $\theta\in\mathbb R^d$, with prior $\varphi$; the target is $f^*\in\mathcal F$, and

$$
K(\theta)=\|f_\theta-f^*\|^2 .
$$

With Gaussian output noise of variance $\tfrac12$ this is exactly the population excess negative log-likelihood; other noise levels rescale $n$. As in Notebook 2, $Z_n=\int\varphi\,e^{-nK}$ and $F_n=-\log Z_n$; if $K_*=\min K>0$, the excess free energy is $\bar F_n=F_n-nK_*$. The effective exponent of Notebook 3 §3.1 (written $\hat\lambda(n)$ in the lecture) is

$$
\lambda_{\rm eff}(n)=\frac{d\bar F_n}{d\log n}=n\,\mathbb E_{\pi_n}[K-K_*]\;\xrightarrow{\;n\to\infty\;}\;\lambda .
$$

The running example is the **cosine network** $f_\theta(x)=\sum_{i=1}^Ha_i\cos(w_ix)$, $\theta=(a,w)\in[-1,1]^{2H}$ with the uniform prior, and **learning zero** means $f^*=0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp, gammaln, log_ndtr, erf, roots_hermitenorm, eval_hermitenorm
from scipy.optimize import linprog

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def slope(F, logn):
    # effective exponent dF/dlog n by finite differences
    return np.gradient(F, logn)


def log_grid(lo=-30.0, m=6000):
    # nodes on (0,1] equally spaced in log; returns the nodes and the log of the quadrature weights
    l = np.linspace(lo, 0.0, m)
    return np.exp(l), l + np.log(l[1] - l[0])

<a id="slt5-s1"></a>

## 5.1 Generic models are regular

*Question:* are degenerate minima stable under small perturbations of the target or the activation?

*Notation:* $\nabla K$, $\nabla^2K$ the gradient and Hessian in $\theta$. A critical point is *non-degenerate* if $\nabla^2K$ is invertible there; $K$ is *Morse* if all its critical points are non-degenerate. For the cosine network, $c_w=\cos(w\,\cdot)\in\mathcal F$.

Learning zero is singular for every activation: whenever $a_i=0$ the weight $w_i$ is unconstrained, so the zero set of $K$ contains whole coordinate planes. This is a coincidence of the target, not of the architecture. Exercise 5.1a shows that such coincidences disappear under almost every perturbation, so that with probability one the learning coefficient of a real model is $d/2$. Exercise 5.1b shows why this is less damning than it sounds: the perturbation sets a *scale*, and below it the singular model is what the free energy sees.

**Exercise 5.1a (Tilting makes a loss Morse).** *Optional: for anyone who hasn't seen why generic functions are non-singular.* You may use *Sard's theorem*: for a smooth map $g:U\to\mathbb R^d$ on an open $U\subset\mathbb R^d$, the set of *critical values* $\{g(\theta):\det dg(\theta)=0\}$ has Lebesgue measure zero.

1. Let $K$ be smooth on $U$ and, for $v\in\mathbb R^d$, $K_v(\theta)=K(\theta)-\langle v,\theta\rangle$. Show that $\theta$ is a degenerate critical point of $K_v$ exactly when $\nabla K(\theta)=v$ and $\det\nabla^2K(\theta)=0$. Conclude that $K_v$ is Morse for almost every $v$.
2. *(General tilts.)* Let $G:U\to\mathbb R^m$ be smooth with $dG(\theta)$ injective for every $\theta$, and $K_v=K-\langle v,G\rangle$, $v\in\mathbb R^m$. Show $K_v$ is Morse for almost every $v$. <details><summary>Hint</summary>Near $\theta_0$ choose $d$ components $G_S$ with $dG_S(\theta_0)$ invertible and use $y=G_S(\theta)$ as local coordinates; for fixed $v_{S^c}$ part 1 applies in $y$. Non-degeneracy of a critical point does not depend on coordinates. Finish with Fubini and a countable cover.</details>
3. *(Perturbing the target.)* Replace $f^*$ by $f^*+\sum_{j=1}^mv_jg_j$ for fixed $g_j\in\mathcal F$. Show that $K$ changes by a tilt with $G_j(\theta)=2\langle f_\theta,g_j\rangle$, plus a constant. Deduce: on a compact set where the tangent functions $\partial f_\theta/\partial\theta_\alpha$ are linearly independent, almost every such perturbation makes all critical points non-degenerate, once $m$ is large and the $g_j$ are the first $m$ elements of an orthonormal basis.
4. *(Dead neurons are not minima.)* For the cosine network and any target, let $\theta$ be a critical point with $a_i=0$, and $r=f_\theta-f^*$. Show that the $(a_i,w_i)$ block of $\nabla^2K$ is $\begin{pmatrix}2\|c_{w_i}\|^2&2b_i\\2b_i&0\end{pmatrix}$ with $b_i=\langle r,\partial_wc_w|_{w=w_i}\rangle$. Conclude that such a point is a saddle or degenerate, and degenerate only if $w\mapsto\langle r,c_w\rangle$ has a double zero at $w_i$. <details><summary>Hint</summary>$\partial_{w_i}f_\theta=a_i\,\partial_wc_{w_i}$ vanishes at $a_i=0$.</details>

**Exercise 5.1b (A small target is learning zero until $n\sim\varepsilon^{-2}$).** *Lower priority: this is the two-plateau picture from the lecture.*

1. *(Sandwich.)* Fix any model and a target with $\|f^*\|=\varepsilon$, and write $F^{(\varepsilon)}_n$, $F^{(0)}_n$ for the free energies with targets $f^*$ and $0$. Show that for $n\varepsilon^2\le1$
$$
F^{(0)}_{n/2}-1\;\le\;F^{(\varepsilon)}_n\;\le\;F^{(0)}_{2n}+2 .
$$
Deduce that if $F^{(0)}_n=\lambda_0\log n+O(1)$, then $\bar F^{(\varepsilon)}_n=\lambda_0\log n+O(1)$ for all $n\le\varepsilon^{-2}$, with constants independent of $\varepsilon$. <details><summary>Hint</summary>$\|f_\theta-f^*\|$ lies between $\|f_\theta\|-\varepsilon$ and $\|f_\theta\|+\varepsilon$; use $(x+y)^2\le2x^2+2y^2$ and $(x-y)^2\ge\tfrac12x^2-y^2$.</details>
2. *(One neuron.)* Take $f_\theta=a\,c_w$ with $(a,w)\in[-1,1]^2$, and target $\varepsilon h$ with $h=\tfrac1{\sqrt2}\big(c_{w_0}/\|c_{w_0}\|+x\big)$, $w_0=0.6$. Show $\langle c_w,c_v\rangle=\tfrac12\big(e^{-(w-v)^2/2}+e^{-(w+v)^2/2}\big)$, $\langle c_w,x\rangle=0$, and
$$
K(a,w)=\big(a\|c_w\|-\varepsilon\rho(w)\big)^2+\varepsilon^2\big(1-\rho(w)^2\big),\qquad\rho(w)=\frac{\langle c_w,h\rangle}{\|c_w\|}.
$$
3. Show $\rho^2\le\tfrac12$ with equality exactly at $w=\pm w_0$. So $K_*=\varepsilon^2/2$, and the Hessian at the minima has one eigenvalue of order $1$ and one of order $\varepsilon^2$. <details><summary>Hint</summary>Cauchy–Schwarz for $\langle c_w,c_{w_0}\rangle$. For the Hessian use the coordinates $(\alpha,w)$ with $\alpha=a\|c_w\|-\varepsilon\rho(w)$: at a critical point the Hessian transforms by congruence.</details>
4. Learning zero with one neuron has $\lambda_0=\tfrac12$. Conclude that $\lambda_{\rm eff}(n)\approx\tfrac12$ for $n\lesssim\varepsilon^{-2}$ and $\lambda_{\rm eff}(n)\to1$ for $n\gg\varepsilon^{-2}$. The code below checks this.

**Effective singularity.** Exercise 5.1b is the general picture in miniature. The generic model is regular; the free energy of the singular model (here learning zero) governs it over the window $n\lesssim\varepsilon^{-2}$, where $\varepsilon$ measures the distance from the singular situation. The rest of this notebook computes free energies in such windows, and Notebook 6 computes the learning coefficient of learning zero with $H$ neurons, which is what governs small targets for a width-$H$ cosine network.

**Code.** $\lambda_{\rm eff}(n)$ for the one-neuron model of Exercise 5.1b. The $a$-integral of $e^{-nK}$ is Gaussian and done exactly; the $w$-integral is a Riemann sum.

In [ ]:
w0 = 0.6
ip = lambda w, v: 0.5 * (np.exp(-(w - v)**2 / 2) + np.exp(-(w + v)**2 / 2))   # <c_w, c_v>
w = np.linspace(-1, 1, 400001); dw = w[1] - w[0]
A = ip(w, w)                                   # ||c_w||^2
B = ip(w, w0) / np.sqrt(2 * ip(w0, w0))        # <c_w, h>
rho2 = B**2 / A


def logZ_small_target(n, eps):
    # log of (1/4) int_{[-1,1]^2} exp(-n K) da dw, with K = A a^2 - 2 eps B a + eps^2
    m, s = eps * B / A, np.sqrt(2 * n * A)    # in a: Gaussian with mean m, sd 1/s
    lo, hi = log_ndtr(s * (-1 - m)), log_ndtr(s * (1 - m))
    log_mass = hi + np.log1p(-np.exp(lo - hi))
    la = -n * eps**2 * (1 - rho2) + 0.5 * np.log(2 * np.pi) - np.log(s) + log_mass
    return logsumexp(la) + np.log(dw / 4)


logn = np.linspace(np.log(10), np.log(1e11), 100)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for eps in [1e-2, 1e-3]:
    Kmin = eps**2 * (1 - rho2.max())
    Fbar = np.array([-logZ_small_target(n, eps) - n * Kmin for n in np.exp(logn)])
    ax.plot(logn / np.log(10), slope(Fbar, logn), label=fr"$\varepsilon={eps:g}$")
    ax.axvline(np.log10(eps**-2), ls=":", c="gray")
ax.set_xlabel(r"$\log_{10} n$"); ax.set_ylabel(r"$\lambda_{\rm eff}(n)$")
ax.set_title(r"one neuron, target $\varepsilon h$ (dotted: $n=\varepsilon^{-2}$)"); ax.legend(); plt.show()

The crossover sits about a decade above $\varepsilon^{-2}$: the posterior width in $w$, $(0.66\,n\varepsilon^2)^{-1/2}$, has to fall below the distance $\approx0.4$ from $w_0$ to the edge of the prior. The overshoot above $1$ just after the crossover is a transient, not a change of learning coefficient.

<a id="slt5-s2"></a>

## 5.2 Function space

*Question:* at sample size $n$, how many directions of function space does the free energy see?

*Notation:* $\mathrm{He}_k$ is the monic polynomial of degree $k$ that is orthogonal, in $\mathcal F$, to all polynomials of lower degree: $\mathrm{He}_0=1$, $\mathrm{He}_1=x$, $\mathrm{He}_2=x^2-1$, $\mathrm{He}_3=x^3-3x$. They satisfy $\langle\mathrm{He}_j,\mathrm{He}_k\rangle=k!\,\delta_{jk}$, so $h_k=\mathrm{He}_k/\sqrt{k!}$ is an orthonormal basis. Write $c_k(\theta)=\langle f_\theta,h_k\rangle$ and $c_k^*=\langle f^*,h_k\rangle$, so that $K(\theta)=\sum_k(c_k(\theta)-c_k^*)^2$. $\mathcal F_{\le k}=\mathrm{span}(h_0,\dots,h_k)$ is the space of polynomials of degree $\le k$, and $K_{\le k}=\sum_{j\le k}(c_j-c_j^*)^2$ is the loss seen through it, with free energy $F_{n,\le k}$ and posterior expectations $\mathbb E^{\le k}_{\pi_n}$.

You may use the identity $\mathbb E_\gamma[f\,\mathrm{He}_k]=\mathbb E_\gamma[f^{(k)}]$ for smooth $f$ whose derivatives grow at most polynomially. (It is $k$ integrations by parts, using $\mathrm{He}_k\,\gamma=(-1)^k\gamma^{(k)}$.)

**Exercise 5.2a (The cosine integral, and truncation).** *The core exercise of this section.*

1. Show that $\langle\cos(w\,\cdot),\mathrm{He}_k\rangle=(-1)^jw^{2j}e^{-w^2/2}$ for $k=2j$, and $0$ for odd $k$. Deduce that for the cosine network
$$
c_{2j}(\theta)=\frac{(-1)^jM_j(\theta)}{\sqrt{(2j)!}},\qquad M_j(\theta)=\sum_{i=1}^Ha_ie^{-w_i^2/2}w_i^{2j},\qquad c_{2j+1}(\theta)=0 .
$$
2. Let $f^*=0$, or any polynomial of degree $\le2k$. Show that $0\le K-K_{\le2k}\le2H^2/(2k+2)!$ on the whole box. <details><summary>Hint</summary>$|M_j|\le H$, and $\sum_{j>k}1/(2j)!\le2/(2k+2)!$.</details>
3. *(The truncation theorem.)* Suppose $(2k+2)!\ge2nH^2$. Show that $|F_n-F_{n,\le2k}|\le1$, and that every observable $A\ge0$ satisfies $e^{-1}\,\mathbb E^{\le2k}_{\pi_n}[A]\le\mathbb E_{\pi_n}[A]\le e\,\mathbb E^{\le2k}_{\pi_n}[A]$. <details><summary>Hint</summary>Write $K=K_{\le2k}+R$. Part 2 gives $e^{-nR}\in[e^{-1},1]$. Compare the partition functions, then the posterior densities.</details>
4. For $H=100$, find the smallest such $k$ at $n=10^6$ and $n=10^{12}$. How does it grow with $n$?

**Code.** Hermite coefficients of $\cos(wx)$ by Gauss quadrature, against the formula of Exercise 5.2a.1; then the smallest $k$ with $(2k+2)!\ge2nH^2$ for $H=100$.

In [ ]:
x, wq = roots_hermitenorm(150); wq = wq / np.sqrt(2 * np.pi)   # Gauss quadrature for E_gamma
for wv in [0.5, 1.0]:
    num = [np.sum(wq * np.cos(wv * x) * eval_hermitenorm(2 * j, x)) / np.exp(0.5 * gammaln(2 * j + 1)) for j in range(5)]
    th = [(-1)**j * np.exp(-wv**2 / 2) * wv**(2 * j) / np.exp(0.5 * gammaln(2 * j + 1)) for j in range(5)]
    print(f"w={wv}: quadrature", np.round(num, 8), "\n        formula   ", np.round(th, 8))

H = 100
for n in [1e2, 1e4, 1e6, 1e9, 1e12, 1e20]:
    k = next(k for k in range(200) if np.log(2 * n * H**2) <= gammaln(2 * k + 3))
    Lg = np.log(n)
    print(f"n=1e{int(np.log10(n)):>2}:  k = {k:2d} (degree {2 * k:2d})    log n / (2 log log n) = {Lg / (2 * np.log(Lg)):.1f}")

**Exercise 5.2b (Other activations: the nearest pole).** *Optional.* The cosine is special: its coordinates decay factorially. For other activations the decay rate is set by the nearest complex singularity of the activation. The idea is to write the activation as a superposition of cosine and sine neurons and use 5.2a.1. Let $\sigma$ be integrable, with Fourier transform $\hat\sigma(\omega)=\int\sigma(x)e^{-i\omega x}dx$, so that $\sigma(x)=\frac1{2\pi}\int\hat\sigma(\omega)e^{i\omega x}d\omega$, and let $c_k=\langle\sigma,h_k\rangle$.

1. Show the formula below. <details><summary>Hint</summary>As in 5.2a.1, $\langle e^{i\omega\cdot},\mathrm{He}_k\rangle=\mathbb E[\partial_x^ke^{i\omega x}]=(i\omega)^ke^{-\omega^2/2}$.</details>
$$
c_k=\frac{i^k}{2\pi\sqrt{k!}}\int_{\mathbb R}\hat\sigma(\omega)\,\omega^ke^{-\omega^2/2}\,d\omega .
$$
2. Let $W_k(\omega)=\omega^ke^{-\omega^2/2}/\sqrt{k!}$ for $\omega>0$. Show that $\log W_k$ is concave with second derivative $\le-1$, maximal at $\omega=\sqrt k$, with maximum value at most $(2\pi k)^{-1/4}$. So $W_k$ is a bump of width $O(1)$ around $\sqrt k$ and of mass $\asymp k^{-1/4}$, and heuristically $|c_k|\approx k^{-1/4}|\hat\sigma(\pm\sqrt k)|$: *the $k$-th Hermite coordinate measures frequency $\sqrt k$.* <details><summary>Hint</summary>Stirling's bound $k!\ge\sqrt{2\pi k}\,(k/e)^k$.</details>
3. *(Analytic in a strip.)* Suppose $|\hat\sigma(\omega)|\le Ce^{-\tau|\omega|}$. Show that $\sigma$ extends to a holomorphic function on the strip $|\mathrm{Im}\,z|<\tau$, and the bound below. <details><summary>Hint</summary>Substitute $v=\omega+\tau$. Use $(v-\tau)^k\le v^ke^{-k\tau/v}$, then $k/v\ge2\sqrt k-v$, then part 2 in the form $v^ke^{-v^2/2}\le(k/e)^{k/2}e^{-(v-\sqrt k)^2/2}$.</details>
$$
\lvert c_k\rvert\le C_\tau\,k^{-1/4}e^{-\tau\sqrt k}.
$$
4. *($\tanh$.)* $\tanh$ is not integrable, but $\tanh'=\mathrm{sech}^2$ is, with Fourier transform $\pi\omega/\sinh(\pi\omega/2)$. Show $\mathbb E[f'\,\mathrm{He}_k]=\mathbb E[f\,\mathrm{He}_{k+1}]$ for bounded smooth $f$ with bounded derivative, so $\langle\tanh,h_{k+1}\rangle=\langle\mathrm{sech}^2,h_k\rangle/\sqrt{k+1}$. Conclude $|\langle\tanh,h_k\rangle|\le e^{-(\pi/2-\delta)\sqrt k}$ for every $\delta>0$ and large $k$. The poles of $\tanh$ nearest the real axis are at $\pm i\pi/2$: the decay rate is the distance from the real axis to the nearest singularity. <details><summary>Hint</summary>$\mathrm{He}_{k+1}=x\,\mathrm{He}_k-\mathrm{He}_k'$, so $(\mathrm{He}_k\gamma)'=-\mathrm{He}_{k+1}\gamma$ for the Gaussian density $\gamma$. Integrate by parts.</details>
5. *(ReLU.)* Apply part 4's identity twice, with $\mathrm{ReLU}''=\delta_0$, to get the exact formula $\langle\mathrm{ReLU},h_{k+2}\rangle=\gamma(0)\,\mathrm{He}_k(0)/\sqrt{k!\,(k+1)(k+2)}$ for $k\ge0$. Using $\mathrm{He}_{2m}(0)=(-1)^m(2m-1)!!$ and $\mathrm{He}_{2m+1}(0)=0$, show $|\langle\mathrm{ReLU},h_k\rangle|\asymp k^{-5/4}$ for even $k$. Is this what part 2's heuristic predicts?
6. Use parts 4–5 to estimate $k_0(n)$, the number of Hermite coordinates needed for a tail $\sum_{k>k_0}c_k^2\le1/n$, for $\tanh$ and for ReLU. Compare with the cosine (Exercise 5.2a.4).

**Code.** Hermite coefficients of $\tanh$ and ReLU by quadrature, using the stable recursion for $h_k\sqrt\gamma$. Left: $-\log|c_k|/\sqrt k$ for $\tanh$, approaching the rate $\pi/2$ of Exercise 5.2b.4 (slowly, because of the polynomial factors). Right: ReLU against the exact formula of 5.2b.5.

In [ ]:
xg = np.linspace(-40, 40, 160001); dxg = xg[1] - xg[0]
def hermite_coeffs(f, kmax):
    # c_k = int f h_k gamma dx = int (f sqrt(gamma)) (h_k sqrt(gamma)) dx; h_k sqrt(gamma) by the three-term recursion
    g = f(xg) * np.exp(-xg**2 / 4) / (2 * np.pi)**0.25
    p0 = np.exp(-xg**2 / 4) / (2 * np.pi)**0.25; p1 = xg * p0
    c = [np.sum(g * p0) * dxg, np.sum(g * p1) * dxg]
    for k in range(1, kmax):
        p0, p1 = p1, (xg * p1 - np.sqrt(k) * p0) / np.sqrt(k + 1)
        c.append(np.sum(g * p1) * dxg)
    return np.array(c)

kmax = 600
c_tanh = hermite_coeffs(np.tanh, kmax)
c_relu = hermite_coeffs(lambda t: np.maximum(t, 0), kmax)
k_odd = np.arange(1, 401, 2); k_even = np.arange(2, 401, 2)
m = (k_even - 2) // 2                           # c_{2m+2} = gamma(0) (-1)^m (2m-1)!! / sqrt((2m)! (2m+1)(2m+2))
log_dfact = gammaln(2 * m + 1) - m * np.log(2) - gammaln(m + 1)        # log (2m-1)!! = log (2m)! - log (2^m m!)
relu_exact = np.exp(-0.5 * np.log(2 * np.pi) + log_dfact - 0.5 * (gammaln(2 * m + 1) + np.log((2 * m + 1) * (2 * m + 2))))

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(k_odd, -np.log(np.abs(c_tanh[k_odd])) / np.sqrt(k_odd), ".", ms=3, label=r"$\tanh$: $-\log|c_k|/\sqrt{k}$")
axs[0].axhline(np.pi / 2, ls="--", c="gray", label=r"$\pi/2$")
axs[0].set_xlabel("$k$ (odd)"); axs[0].set_ylim(1, 1.8); axs[0].legend(fontsize=8)
axs[1].loglog(k_even, np.abs(c_relu[k_even]), ".", ms=3, label="ReLU, quadrature")
axs[1].loglog(k_even, relu_exact, "--", label=r"exact formula (5.2b.5)")
axs[1].loglog(k_even, 0.4 * k_even**-1.25, ":", c="gray", label=r"$\propto k^{-5/4}$")
axs[1].set_xlabel("$k$"); axs[1].legend(fontsize=8); plt.tight_layout(); plt.show()

for name, c in [("tanh", c_tanh), ("cos(x)", np.array([(-1)**(k // 2) * np.exp(-0.5 - 0.5 * gammaln(k + 1)) * (k % 2 == 0) for k in range(kmax + 1)]))]:
    tail = np.cumsum((c**2)[::-1])[::-1]
    for e in (6, 12):
        k0 = next(k for k in range(kmax) if tail[k + 1] <= 10.0**-e)
        print(f"{name:7s} n=1e{e:<2d}: Hermite index k0 = {k0:3d}, nonzero coordinates = {int(np.sum(np.abs(c[:k0 + 1]) > 1e-15))}")

**Further problems.** *Low priority: do §5.3 first.*

**Exercise 5.2c (A field space that keeps growing).** A toy with exact formulas: coordinates $u_k$, $k\ge0$, loss $K(u)=\sum_k\mu_ku_k^2$ with $\mu_k=1/k!$, and a standard Gaussian prior on each $u_k$ (instead of a uniform one, for closed forms).

1. Show $F_n=\tfrac12\sum_k\log(1+2n\mu_k)$ and $\lambda_{\rm eff}(n)=\sum_k\frac{n\mu_k}{1+2n\mu_k}$. <details><summary>Hint</summary>$\int e^{-n\mu u^2}\,d\mathcal N(0,1)(u)=(1+2n\mu)^{-1/2}$.</details>
2. Let $k_0(n)=\#\{k:2n\mu_k\ge1\}$. Show $\lambda_{\rm eff}(n)=\tfrac12k_0(n)+O(1)$, while $F_n=\tfrac14k_0(n)\log n\,(1+o(1))$ (the $o(1)$ decays slowly; see the plot). So the slope and the ratio $F_n/\log n$ differ by a factor of $2$. <details><summary>Hint</summary>$n\mu_k$ changes superexponentially in $k$, so only $O(1)$ terms of $\lambda_{\rm eff}$ are neither near $\tfrac12$ nor near $0$. For $F_n$ use $\sum_{k<k_0}\log k!\approx\tfrac12k_0^2\log k_0$ and $k_0\log k_0\approx\log n$.</details>
3. Interpret. At every $n$ the model looks regular with about $k_0(n)$ active directions, but $\lambda_{\rm eff}$ never settles: it grows like $\log n/(2\log\log n)$. Where does the factor $2$ come from?

With $\mu_j=1/(2j)!$, the weights of learning zero in Exercise 5.2a, the same holds with $k_0$ halved.

In [ ]:
logn = np.linspace(np.log(10), np.log(1e40), 400); n = np.exp(logn)
lmu = -gammaln(np.arange(80) + 1)                         # log mu_k = -log k!
F = 0.5 * np.logaddexp(0, np.log(2 * n)[:, None] + lmu[None, :]).sum(1)
k0 = (np.log(2 * n)[:, None] + lmu[None, :] >= 0).sum(1)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(logn / np.log(10), slope(F, logn), label=r"$\lambda_{\rm eff}(n)=dF_n/d\log n$")
ax.plot(logn / np.log(10), k0 / 2, ls="--", label=r"$k_0(n)/2$")
ax.plot(logn / np.log(10), F / logn, label=r"$F_n/\log n$")
ax.set_xlabel(r"$\log_{10}n$"); ax.legend(); ax.set_title(r"$\mu_k=1/k!$"); plt.show()

**Exercise 5.2d (Upper bounds from a submersion point).**

1. Let $\Phi:U\to\mathbb R^N$ be smooth on an open $U\subset\mathbb R^d$, $K=|\Phi|^2$, and suppose $\Phi(\theta^*)=0$ with $d\Phi(\theta^*)$ surjective and $\varphi$ bounded below near $\theta^*$. Show $V(t)\ge c\,t^{N/2}$ for small $t$, hence $\lambda\le N/2$. <details><summary>Hint</summary>The implicit function theorem gives local coordinates $(\xi,\zeta)\in\mathbb R^N\times\mathbb R^{d-N}$ in which $\Phi=\xi$.</details>
2. Surjectivity matters: for $\Phi(\theta)=|\theta|^2$ on $\mathbb R^d$ ($N=1$), compute $\lambda$.
3. Learning zero truncated to $\mathcal F_{\le2k}$ is $K_{\le2k}=|\Phi_k|^2$ with $\Phi_k=\big(M_j/\sqrt{(2j)!}\big)_{j=0}^k$. Show that $d\Phi_k$ is surjective at every point with $a=0$ and $w_1^2,\dots,w_H^2$ distinct, provided $k+1\le H$. Conclude $\lambda_{\le2k}\le(k+1)/2$. <details><summary>Hint</summary>At $a=0$, $\partial M_j/\partial a_i=e^{-w_i^2/2}w_i^{2j}$.</details>
4. *(Where the asymptotics start.)* Part 3 is a statement about $t\to0$ at fixed $k$. Show that for $t\ge2H^2/(2k)!$, $\{K_{\le2k-2}<t/2\}\subset\{K_{\le2k}<t\}$, so $V_{\le2k}(t)\ge V_{\le2k-2}(t/2)$ and the behaviour $V_{\le2k}(t)\asymp t^{(k+1)/2}$ can only set in for $t\lesssim H^2/(2k)!$. With $k=k_0(n)$ and $t=1/n$, where does $n$ sit relative to this regime? Compare Exercise 5.2c, where direction $k$ becomes active at $n\approx1/\mu_k$.

<a id="slt5-s3"></a>

## 5.3 Newton polygons

*Question:* what does a coefficient comparable to $1/n$ do to the free energy?

*Notation:* $V(t)$ is the prior volume of $\{K<t\}$. Notebook 2 Exercise 2.1.1 gives $Z_n=n\int_0^\infty e^{-nt}V(t)\,dt$.

Notebook 3 §3.1 treated $w^4+\delta w^2$: the effective exponent moves from $\tfrac14$ to $\tfrac12$ around $n\approx\delta^{-2}$. Here is the general one-parameter version. It is the basic example of the lecture's bad news: once a coefficient is comparable to $1/n$, no single learning coefficient describes the model, but the free energy is still computable, and it has a simple shape.

**Exercise 5.3a (Newton polygons in one parameter).** *The core exercise of this section.* Uniform prior on $a\in[-1,1]$, and
$$
K(a)=\sum_{j=1}^Jc_j\,a^{2j},\qquad c_j\ge0,\quad c_J=1 .
$$
The *Newton polygon* of $K$ is the lower convex hull of the points $(2j,-\log c_j)$.

1. Show that $K$ lies between its largest term and $J$ times it. Deduce that $\ell(t/J)\le V(t)\le\ell(t)$, where $\ell(t)=\min\big(1,\ \min_j(t/c_j)^{1/(2j)}\big)$.
2. Deduce the formula below. <details><summary>Hint</summary>$\ell(st)\le s^{1/2}\ell(t)$ for $s\ge1$. Split the integral for $Z_n$ at $t=1/n$.</details>
$$
F_n=\max_j\frac{\log n+\log c_j}{2j}+O_J(1).
$$
3. *(The simplest case.)* For $K=\varepsilon a^2+a^4$ this says $F_n=\max\big(\tfrac14\log n,\ \tfrac12\log n+\tfrac12\log\varepsilon\big)+O(1)$, uniformly in $\varepsilon\in(0,1]$. Where does $\lambda_{\rm eff}$ jump? What does the free energy remember about $\varepsilon$ above the jump, and below it?
4. Show that as $n$ grows, $\lambda_{\rm eff}$ runs through the values $\tfrac1{2j}$ for exactly those $j$ that are vertices of the Newton polygon, switching between consecutive vertices $j<j'$ at $\log n_*=\frac{j\log c_{j'}-j'\log c_j}{j'-j}$. For $K=\varepsilon^6a^2+\varepsilon^2a^4+a^6$, find both switches. Then replace $\varepsilon^2a^4$ by $\varepsilon^4a^4$ and show that the $a^4$ term never governs.

Formulas like part 2's, maxima of linear functions of logarithms, are tropical polynomials. The optional §5.4 develops the arithmetic behind them.

**Exercise 5.3b (Two dimensions, and where tropical counting stops).**

1. One neuron $a\,\sigma(wx)$ with $\sigma(z)=\varepsilon+z$, learning zero, uniform prior on $[-1,1]^2$. Show $K=a^2(\varepsilon^2+w^2)$ exactly.
2. Show $V(t)\asymp\sqrt t\,\big(1+\min(\log\tfrac1\varepsilon,\ \tfrac12\log\tfrac1t)\big)$ for $t\le\tfrac12$, and deduce
$$
\bar F_n=\tfrac12\log n-\log\Big(1+\min\big(\log\tfrac1\varepsilon,\ \tfrac12\log n\big)\Big)+O(1).
$$
At $\varepsilon=0$ this is $\tfrac12\log n-\log\log n$ ($\lambda=\tfrac12$, $m=2$). For $\varepsilon>0$, $\lambda=\tfrac12$ and eventually $m=1$, but the $\log\log n$ term saturates at $\log\log(1/\varepsilon)$ once $n\gtrsim\varepsilon^{-2}$: here the scale enters through the multiplicity term rather than as a multiple of $\log\varepsilon$. <details><summary>Hint</summary>For fixed $w$, the $a$-section of $\{K<t\}$ has half-length $\min\big(1,\sqrt{t/(\varepsilon^2+w^2)}\big)$. Split at $w^2=t$ when $t>\varepsilon^2$.</details>
3. *(Tropical reading.)* With $\ell_a=\log|a|$, $\ell_w=\log|w|$, show $\log K=2\ell_a+\max(2\log\varepsilon,2\ell_w)+O(1)$ and
$$
V(t)\asymp\int_{\{2\ell_a+\max(2\log\varepsilon,\,2\ell_w)<\log t,\ \ell_a,\ell_w\le0\}}e^{\ell_a+\ell_w}\,d\ell_a\,d\ell_w .
$$
Show that the maximum of $\ell_a+\ell_w$ over this polyhedron is $\tfrac12\log t$, attained on a segment of length $\min(\log\tfrac1\varepsilon,\tfrac12\log\tfrac1t)$, and that this length is the logarithmic factor in part 2. The general principle: the exponent of $V(t)$ is the value of a linear program, and the power of $\log(1/t)$ is the dimension of the face on which the optimum is attained; a scale such as $\log\varepsilon$ can truncate that face. <details><summary>Hint</summary>For fixed $\ell_w$, integrate $e^{\ell_a}$ up to the boundary first.</details>
4. *(Cancellation.)* For $K=(a-b)^2$ on $[-1,1]^2$, the same recipe applied to the monomials $a^2,ab,b^2$ with signs ignored predicts $\log K\approx2\max(\ell_a,\ell_b)$ and $\lambda=1$. Show that the true value is $\lambda=\tfrac12$.

Tropical counting is valid up to constants when $K$ is a sum of monomials with nonnegative coefficients, or more generally when the monomials cannot cancel. Learning zero with $H$ neurons is the opposite case: $K=\sum_jM_j^2/(2j)!$, and each moment $M_j=\sum_i\tilde a_iw_i^{2j}$ is a sum over neurons that cancels exactly on the zero set. Notebook 6 removes this cancellation by passing to a dual problem in which neurons enter through a product.

**Code.** Left: $\bar F_n$ for the three-term loss of Exercise 5.3a.4 with $\varepsilon=0.1$, against the maximum of three lines. Right: $\bar F_n-\tfrac12\log n$ for $K=a^2(\varepsilon^2+w^2)$ against Exercise 5.3b.2 (up to a constant).

In [ ]:
logn = np.linspace(np.log(10), np.log(1e14), 140); ns = np.exp(logn)
a, la = log_grid()
eps = 0.1
s = np.log([eps**6, eps**2, 1.0])                        # s_j = log c_j, j = 1, 2, 3
K = eps**6 * a**2 + eps**2 * a**4 + a**6
F = np.array([-logsumexp(-n * K + la) for n in ns])
pred = np.max([(logn + s[j - 1]) / (2 * j) for j in (1, 2, 3)], axis=0)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(logn / np.log(10), F, label=r"$\bar F_n$")
axs[0].plot(logn / np.log(10), pred + (F - pred)[-1], "--", label=r"$\max_j\,(\log n+s_j)/2j$ + const")
for x in (6, 10):
    axs[0].axvline(x, ls=":", c="gray")
axs[0].set_title(r"$K=\varepsilon^6a^2+\varepsilon^2a^4+a^6$, $\varepsilon=0.1$ (dotted: $n=\varepsilon^{-6},\varepsilon^{-10}$)", fontsize=9)
axs[0].set_xlabel(r"$\log_{10}n$"); axs[0].legend(fontsize=8)

w, lw = log_grid()
logn2 = np.linspace(np.log(10), np.log(1e12), 120)
for eps in [1e-2, 1e-4]:
    s2 = eps**2 + w**2
    # int_0^1 exp(-n a^2 s) da = sqrt(pi/(4 n s)) erf(sqrt(n s)); then integrate over w on the log grid
    F2 = np.array([-logsumexp(np.log(np.sqrt(np.pi / (4 * n * s2)) * erf(np.sqrt(n * s2))) + lw) for n in np.exp(logn2)])
    G = F2 - 0.5 * logn2
    pred2 = -np.log(1 + np.minimum(np.log(1 / eps), 0.5 * logn2))
    axs[1].plot(logn2 / np.log(10), G, label=fr"$\varepsilon={eps:g}$")
    axs[1].plot(logn2 / np.log(10), pred2 + (G - pred2)[-1], "--", c=axs[1].lines[-1].get_color())
axs[1].set_title(r"$K=a^2(\varepsilon^2+w^2)$: $\bar F_n-\frac{1}{2}\log n$ (dashed: prediction $+$ const)", fontsize=9)
axs[1].set_xlabel(r"$\log_{10}n$"); axs[1].legend(fontsize=8); plt.tight_layout(); plt.show()

**Exercise 5.3c (Furman's cubic model at finite $n$).** This is Exercises 2 and 12 of Zach Furman's [*Singular learning theory: exercises*](https://www.lesswrong.com/posts/3HYqTAi4kD35G3BzQ/singular-learning-theory-exercises), read through the Newton polygon. The data are $y\sim\mathcal N(\mu_0^3,1)$ and the model is $\mathcal N(\mu^3,1)$ with $\mu$ uniform on $[-1,1]$, so $K(\mu)=\tfrac12(\mu^3-\mu_0^3)^2$. Take $0<\mu_0\le\tfrac12$. Asymptotically $\lambda=\tfrac12$, while at $\mu_0=0$, $\lambda=\tfrac16$ (Notebook 2 Exercise 2.2.3). Furman's Exercise 12d observes that at $\mu_0=0.1$ and $n\le10^4$ the free energy looks like the $\mu_0=0$ case.

1. With $\delta=\mu-\mu_0$, show $\mu^3-\mu_0^3=\delta(\mu^2+\mu\mu_0+\mu_0^2)$, and that $K\asymp\delta^2(\delta^2+\mu_0^2)^2\asymp\mu_0^4\delta^2+\delta^6$ with absolute constants. <details><summary>Hint</summary>$|\mu\mu_0|\le\tfrac12(\mu^2+\mu_0^2)$; then compare $\mu^2+\mu_0^2$ with $\delta^2+\mu_0^2$.</details>
2. Deduce, uniformly in $\mu_0\in(0,\tfrac12]$,
$$
\bar F_n=\max\big(\tfrac16\log n,\ \tfrac12\log n+2\log\mu_0\big)+O(1),
$$
with the crossover at $n\approx\mu_0^{-6}$, which is $10^6$ for Furman's $\mu_0=0.1$. Expanding $K$ also produces a $\mu_0^2\delta^4$ term: where does its point lie relative to the Newton polygon?
3. In the coordinate $\theta=\mu^3$ the model is the regular normal-mean model, $K=\tfrac12(\theta-\theta_0)^2$. What is the prior density in $\theta$? Rederive part 2 from this form. Where is the singularity now?
4. \* The code below shows $\lambda_{\rm eff}$ overshooting $\tfrac12$ before it settles, peaking at about $0.86$ near $n\approx4.5\,\mu_0^{-6}$, with the same height for both values of $\mu_0$. Why does this not contradict part 2, where does it come from, and why does the peak not depend on $\mu_0$? <details><summary>Hint</summary>$\lambda_{\rm eff}=n\,\mathbb E_{\pi_n}[K]$. In the coordinate $\theta$, where is the posterior mass just after the crossover, and how large is $nK$ there?</details>

In [ ]:
from scipy.integrate import quad

def lam_eff_cubic(n, mu0):
    # n E[K] under the posterior, K = (mu^3 - mu0^3)^2 / 2, mu uniform on [-1, 1]
    K = lambda u: 0.5 * (u**3 - mu0**3)**2
    w = 1 / (3 * mu0**2 * np.sqrt(n))                  # posterior width around mu0 once resolved; tell quad where to look
    pts = sorted({0.0, mu0} | {mu0 + c * w for c in (-10, -3, -1, 1, 3, 10) if -1 < mu0 + c * w < 1})
    kw = dict(points=pts, limit=500, epsabs=0, epsrel=1e-10)
    Z = quad(lambda u: np.exp(-n * K(u)), -1, 1, **kw)[0]
    E = quad(lambda u: n * K(u) * np.exp(-n * K(u)), -1, 1, **kw)[0]
    return E / Z

logn = np.linspace(np.log(10), np.log(1e10), 181)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for mu0 in (0.1, 0.3):
    ax.plot(logn / np.log(10), [lam_eff_cubic(n, mu0) for n in np.exp(logn)], label=fr"$\mu_0={mu0}$")
    ax.axvline(-6 * np.log10(mu0), ls=":", c=ax.lines[-1].get_color())
for v in (1 / 6, 1 / 2):
    ax.axhline(v, c="gray", lw=0.8)
ax.set_xlabel(r"$\log_{10}n$"); ax.set_ylabel(r"$\lambda_{\rm eff}(n)$")
ax.set_title(r"$K=\frac{1}{2}(\mu^3-\mu_0^3)^2$ (dotted: $n=\mu_0^{-6}$)"); ax.legend(); plt.show()

<a id="slt5-s4"></a>

## 5.4 Optional: tropical arithmetic

*Question:* what algebra do exponents of $n$ obey?

*Notation:* for a positive quantity $A$ depending on $n$, its *exponent* is $\log A/\log n$. In this section $\gamma$ denotes an exponent, not the Gaussian density.

§5.3 (and Notebook 6) compute free energies to $O(1)$, so they only track exponents. This section makes the rules explicit. Exercise 5.4c is also the bookkeeping behind the staircases of Notebook 6, Exercise 6.1d.

**Exercise 5.4a (Exponents add by taking the max).**

1. For $x,y\in\mathbb R$ and $n\ge2$, show
$$
\max(x,y)\log n\;\le\;\log(n^x+n^y)\;\le\;\max(x,y)\log n+\log2 .
$$
So on exponents, sums become maxima and products become sums, with an error $O(1)$ that is $O(1/\log n)$ in units of $\log n$. The *tropical semiring* is $\mathbb R\cup\{-\infty\}$ with $x\oplus y=\max(x,y)$ and $x\odot y=x+y$. Check that $\odot$ distributes over $\oplus$, that $-\infty$ is the unit for $\oplus$, and that nothing has an inverse for $\oplus$.
2. Show $\log\sum_{j\le J}c_jn^{x_j}=\max_j\big(x_j\log n+\log c_j\big)+O(\log J)$ for $c_j>0$. A fixed constant $c_j$ shifts the free energy by $O(1)$ and never changes an exponent; it matters for exponents only when it is itself a power of $n$. Relate this to the phrase "coefficients comparable to $1/n$".
3. *(Two basins.)* Notebook 2 Exercise 2.3 has $F_n=-\log\big(e^{-F_n(U)}+e^{-F_n(V)}\big)$. Show $|F_n-\min(F_n(U),F_n(V))|\le\log2$: the free energy of a union is the tropical sum of the free energies, and a phase transition is a change of the active term.

**Exercise 5.4b (Volumes are tropical integrals).**

1. Let $\phi:[0,\infty)\to\mathbb R$ be convex and piecewise linear with finitely many pieces, and suppose its minimum is attained at a single point with nonzero slopes on both sides (or at $0$, with positive slope). Show
$$
\int_0^\infty n^{-\phi(s)}ds=n^{-\min\phi}\,\Theta\Big(\frac1{\log n}\Big).
$$
If instead $\phi$ is minimal on a segment of length $\ell>0$, the integral is $n^{-\min\phi}(\ell+O(1/\log n))$.
2. With $|a|=n^{-s}$ we have $da=(\log n)\,n^{-s}ds$. Use part 1 in two variables to redo the monomial rule of Notebook 2 Exercise 2.2.2 for $K=a^{2k_1}b^{2k_2}$ on $[-1,1]^2$, including the multiplicity. The power of $\log n$ in $V(1/n)$ is the dimension of the set where the linear objective is minimised, as in Exercise 5.3b.3.
3. *(A scale as an exponent.)* Let $K=\varepsilon a^2+a^4$ with $\varepsilon=n^{-\gamma}$, $\gamma\ge0$. With $|a|=n^{-s}$, the exponent of $K$ is $\max(-\gamma-2s,-4s)$. Show that $\{K<1/n\}$ is, up to constants, $\{s>s_*\}$ with $s_*=\max\big(\tfrac14,\tfrac{1-\gamma}2\big)$, and that $\bar F_n/\log n\to s_*$. Sketch $s_*$ as a function of $\gamma$. Where is the kink, and what is it in terms of $n$ and $\varepsilon$? For general $K=\sum_jc_ja^{2j}$ with $c_j=n^{-\gamma_j}$, show $\bar F_n/\log n\to\max_j(1-\gamma_j)/(2j)$, which is Proposition 2 of the lecture (Exercise 5.3a.2).

**Exercise 5.4c (Sizes of roots from the Newton polygon).**

1. Let $p(u)=\sum_{j=0}^dc_ju^j$ with complex $c_j$, $|c_j|=n^{g_j}$ ($g_j=-\infty$ if $c_j=0$; $c_0,c_d\ne0$). On the circle $|u|=n^\rho$, term $j$ has size $n^{g_j+j\rho}$. Suppose a single $j_*$ maximises $g_j+j\rho$. Show that for large $n$, $p$ has exactly $j_*$ roots in $|u|<n^\rho$. <details><summary>Hint</summary>Use this fact from complex analysis: if $|f-g|<|g|$ on a circle, then $f$ and $g$ have the same number of zeros inside it. Take $g=c_{j_*}u^{j_*}$.</details>
2. Deduce that the exponents $\log|u|/\log n$ of the roots of $p$ are, up to $o(1)$, the numbers $-\sigma$ for the slopes $\sigma$ of the edges of the upper convex hull of the points $(j,g_j)$, with as many roots as the edge's horizontal length. Check on $p(u)=n^{-1}u^2+u+1$.
3. The staircases of Notebook 6, Exercise 6.1d.2 are $P_\eta(u)=\sum_{j<N}\eta_ju^j$ with $\eta_0=1$, $\eta_j=r^{\min(1,j/k)}$, $r=e^L$. Use part 2 with $r$ in place of $n$ to show that $P_\eta$ has $k$ roots of size $\approx r^{-1/k}$ and $N-1-k$ roots of size $\approx1$. This is the precise sense in which a staircase "vanishes to order $k$ at $u=0$ at resolution $u_*=r^{-1/k}$". Notebook 6, Exercise 6.1f moves such a cluster of roots to a point $u_0\ne0$.

**Exercise 5.4d (Signs: the tropical answer is one-sided).**

1. Let $K=\sum_{j\le J}m_j(\theta)\ge0$, where the $m_j$ are monomials of either sign, and $T(\theta)=\max_j|m_j(\theta)|$. Show $\{T<t/J\}\subset\{K<t\}$. Conclude that the learning coefficient computed from $T$, which is what counting monomials by size gives, is an *upper* bound for the true one. Check on $K=(a-b)^2$ (Exercise 5.3b.4).
2. In Notebook 6, Exercise 6.1d, the size estimate is applied inside $\chi$, to $|P_\eta|$. Show that ignoring cancellation in $P_\eta$ can only make the predicted $\lambda_H$ too large. This is why the construction of 6.1b needs no extra work, while the lower bound of 6.1e is conditional on 6.1f.